In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print(f'Device          : {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
from datasets.prepare_data import prepare_dataset

# Pass path to Kaggle CSV if you have it:
# splits = prepare_dataset('/path/to/fake_job_postings.csv')
splits = prepare_dataset()   # generates synthetic fallback

train_df = splits['train']
val_df   = splits['val']
test_df  = splits['test']

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
train_df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Class distribution
train_df['label_name'].value_counts().plot.bar(ax=axes[0], color=['#2ecc71','#e74c3c'], rot=0)
axes[0].set_title('Class Distribution (train)')

# Text length
train_df['text_len'] = train_df['text'].str.len()
for lbl, color in [('Real','#2ecc71'), ('Fake','#e74c3c')]:
    subset = train_df[train_df['label_name'] == lbl]['text_len']
    axes[1].hist(subset, bins=40, alpha=0.6, color=color, label=lbl)
axes[1].set_title('Text Length by Class')
axes[1].legend()

# Average text length
train_df.groupby('label_name')['text_len'].mean().plot.bar(
    ax=axes[2], color=['#2ecc71','#e74c3c'], rot=0)
axes[2].set_title('Mean Text Length')

plt.tight_layout()
plt.show()

In [ ]:
from models.train_model import train_pipeline

metrics = train_pipeline()
print('\nFinal metrics:', metrics)

In [ ]:
from models.predictor import FakeJobPredictor

predictor = FakeJobPredictor()

tests = [
    'Software Engineer at TechCorp. $120k salary. Apply with resume.',
    'EARN $5000/WEEK!!! Registration fee $99. No interview. WhatsApp now!!!',
]

for text in tests:
    r = predictor.predict(text)
    print(f'\nText     : {text[:60]}...')
    print(f'Verdict  : {r["prediction"]} ({r["scam_probability"]:.2%})')
    print(f'Keywords : {r["keywords"]}')